# Figure 3 - Variable importance ranking across AGB ranges (2005 and 2015)

Stacked bars show the share (%) of random-forest variable importance within each AGB range (0-200, 0-50, 50-100, 100-150, 150-200 Mg ha$^{-1}$). Inputs: `data/RF_importance_2005.csv`, `data/RF_importance_2015.csv`. Outputs: `output/Figure3_variable_importance.pdf` / `.png`.

Run from this folder (the notebook uses relative paths): `jupyter nbconvert --to notebook --execute Figure3_variable_importance.ipynb`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
from pathlib import Path
import seaborn as sns          # optional; only used for the white style

# ───────────────────────────────
# 0. Global layout and fonts
# ───────────────────────────────
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":          13,
    "axes.titlesize":     15,
    "axes.labelsize":     13,
    "axes.linewidth":     0.4,
    "xtick.major.width":  0.4,
    "ytick.major.width":  0.4,
    "axes.facecolor":     "white",
    "grid.linewidth":     0,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})
sns.set_style("white")         # comment out if seaborn is not installed

# ───────────────────────────────
# 1. Input files (relative to this notebook's folder)
# ───────────────────────────────
file_2005 = Path("data/RF_importance_2005.csv")
file_2015 = Path("data/RF_importance_2015.csv")
assert file_2005.exists() and file_2015.exists(), "Input CSV files not found"

# ───────────────────────────────
# 2. Variable order and fixed colours
# ───────────────────────────────
index_master = [
    "Slope", "HV", "HV_Texture", "HH", "HH_Texture",
    "B3", "B4", "Optical_Bands",
    "NDMI", "NDWI", "LSWI",
    "EVI", "MSAVI", "NDVI"
]
base_colors = [
    "#9c1f53", "#d55c69", "#e66372", "#f99f84", "#ffd876",
    "#ffe5a3", "#e9f5aa", "#b5ddb0",
    "#a6d1c9", "#3db2ab", "#64a4e4",
    "#559be2", "#4d91bd", "#3d8bc0"
]
palette_global = dict(zip(index_master, base_colors))

# ───────────────────────────────
# 3. Pre-processing
# ───────────────────────────────
range_order = ["0-200", "0-50", "50-100", "100-150", "150-200"]

order05 = ["Slope","HV","HV_Texture","HH","HH_Texture",
           "Optical_Bands","NDMI","EVI","MSAVI","NDVI"]
order15 = ["Slope","HV","HV_Texture","HH",
           "B3","B4","NDWI","LSWI","NDVI"]

def prep_df(path, idx_order):
    df = pd.read_csv(path)
    # lower-case 'slope' -> 'Slope'
    df["index"] = df["index"].replace({"slope": "Slope"})
    df["Range"] = pd.Categorical(df["Range"],
                                 categories=range_order, ordered=True)
    full_idx = list(idx_order) + [i for i in df["index"].unique()
                                  if i not in idx_order]
    df["index"] = pd.Categorical(df["index"],
                                 categories=full_idx, ordered=True)
    return df

def to_pct(df):
    # share (%) of each variable's importance within an AGB range
    p = df.pivot_table(index="Range", columns="index",
                       values="important", aggfunc="sum")
    return p.div(p.sum(axis=1), axis=0).fillna(0) * 100

pct05 = to_pct(prep_df(file_2005, order05))
pct15 = to_pct(prep_df(file_2015, order15))

pal05 = {k: palette_global[k] for k in pct05.columns}
pal15 = {k: palette_global[k] for k in pct15.columns}

# ───────────────────────────────
# 4. Plot
# ───────────────────────────────
BAR_W = 0.75
fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

def stacked(df, ax, label, pal):
    bottom = pd.Series(0, index=df.index, dtype=float)
    for col in df.columns:
        ax.bar(df.index, df[col], bottom=bottom,
               width=BAR_W, color=pal[col],
               edgecolor="black", linewidth=0.25)
        bottom += df[col]
    ax.set_title(label, loc="left", fontweight="bold")
    ax.set_ylabel("Importance (%)")
    ax.set_xlabel("AGB Density (Mg ha$^{-1}$)")
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_yticklabels([f"{t}%" for t in ax.get_yticks()])
    ax.tick_params(axis="x", rotation=35, pad=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

stacked(pct05, axes[0], "(a)", pal05)
stacked(pct15, axes[1], "(b)", pal15)

# Legend
handles = [Patch(facecolor=palette_global[i], edgecolor="black",
                 linewidth=0.25, label=i) for i in index_master]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.01, 1),
           frameon=False, fontsize=13, title="Index Bands", title_fontsize=14)

fig.tight_layout(rect=[0, 0, 0.85, 1])   # leave the right 15 % for the legend

# ───────────────────────────────
# 5. Export
# ───────────────────────────────
Path("output").mkdir(exist_ok=True)
fig.savefig("output/Figure3_variable_importance.pdf")         # vector
fig.savefig("output/Figure3_variable_importance.png", dpi=600)
plt.show()
